# Analyze uncertainty after Loop 1 (old measurements)

Collects the optimized inverse-design samples, evaluates targets/uncertainty, filters by cell-type proportion, and plots the Sakurai pattern heatmap against the old measurements.

In [1]:
from pathlib import Path 
import os 
import sys
sys.path.insert(0, "../../../../../../shared_utils")
from experiment_utils import manual_filtering

import matplotlib.pyplot as plt 
import scanpy as sc
import numpy as np 
import pandas as pd
import scipy as sp 
import seaborn as sns
from tqdm import tqdm 
import copy
import math
import yaml
import fpsample

## Read annotation file

In [2]:
ANNOTATION_CFG_FILE = "../../../../../../inverse/loss_guidance/config/annotation/default.yaml"
with open(ANNOTATION_CFG_FILE, "r") as fb:
    annotation_cfg = yaml.safe_load(fb)
protocol_cols = annotation_cfg["protocol_columns"]

ANNOTATION_CFG_FILE_BOUNDS = "../../../../../../inverse/loss_guidance/config/constraints/default_all_cols.yaml"
with open(ANNOTATION_CFG_FILE_BOUNDS, "r") as fb:
    annotation_cfg_bound = yaml.safe_load(fb)
ubound_dict, lbound_dict = annotation_cfg_bound["ubound_dict"],  annotation_cfg_bound["lbound_dict"]

# Collect per parameter bounds 
bounds = {}
for param in lbound_dict:
    bounds[param] = (lbound_dict[param], ubound_dict[param])

### Collect optimized samples 

In [3]:
path_to_results = Path("../../../../../../project_folder/output/inverse/loss_guidance/bloodplus_inverse_fm_old_measurements_official")
paths = os.listdir(path_to_results)

In [4]:
cell_types = ["late_MgkPro"]

In [ ]:
pure_cell_type_solution_dict = {cell_type: [] for cell_type in cell_types}

for experiment_folder in tqdm(paths):
    # Only consider pure populations 
    cell_type_file_paths = path_to_results / experiment_folder
    # For all cell types 
    for cell_type in os.listdir(cell_type_file_paths):
        config_exp_folders = cell_type_file_paths / cell_type
        # For all the experiments in the folders 
        for config_exp_folder in os.listdir(config_exp_folders):
            # Be sure candidates are there 
            if "candidates.csv" in os.listdir(config_exp_folders / config_exp_folder):
                uncertainty_path = config_exp_folders / config_exp_folder / "uncertainty_annotation" / "candidates_with_uncertainties.csv"
                # Be sure uncertainty annotation exists 
                candidate_with_uncertainty_df = pd.read_csv(uncertainty_path)
                candidate_with_uncertainty_df["run_name"] = [file.split("/")[-1] for file in candidate_with_uncertainty_df["cfg:paths:dump_dir"]]
                pure_cell_type_solution_dict[cell_type].append(candidate_with_uncertainty_df)

 67%|██████▋   | 4/6 [00:19<00:07,  3.76s/it]

In [ ]:
# Concatenate data frame 
pure_cell_type_solution_dict_concat = {
    key: pd.concat(val, axis=0) for key, val in pure_cell_type_solution_dict.items()
}

In [ ]:
pure_cell_type_solution_dict_concat["late_MgkPro"]["late_MgkPro_prop"].max()

In [ ]:
pure_cell_type_solution_dict_concat["late_MgkPro"].reset_index(drop=True)

### Initialize targets

Only 4 types were optimized 

In [ ]:
cell_types = ["late_MgkPro"]

In [ ]:
# Cell type fractions
celltype_fraction = {"late_MgkPro": 0.10}

columns_of_interest = [
    'gm-csf_[ng_ml]',
    'tpo_[ng_ml]',
    'sr1_[nm]',
    'um171_[nm]',
    'um729_[µm]',
    'scf_[ng_ml]',
    'butyzamide_[nm]',
    'retinoic_acid_[µm]',
    'ldl_[ng_ml]',
    'il3_[ng_ml]',
    'o2_[%]',
    'days_of_culture',
    'run_name',
    "cfg:constraints:c_scheduler_kwargs:t_warmup",
    "cfg:constraints:c_scheduler_kwargs:vmin",
    "cfg:constraints:c_scheduler_kwargs:vmax",
    "loss", 
    "target_ct_loss_std",
    "target_ct_loss_mean",
    "ct_prop_total_variance",
    "exploitation_score",
    "exploration_score",
    "weighted_uncertainty_score",
    "metric_response_surface", 
    # "indices",
    # "losses",
    # "acq_values"
]

### Filter values

In [ ]:
# Collect filtered and annotated data frames 
filtered_df, annotated_df = manual_filtering(
    df_dict=pure_cell_type_solution_dict_concat,
    bounds=bounds,
    columns_to_keep=columns_of_interest, 
    celltype_fraction=celltype_fraction,
    margin_frac_bounds=0.5, 
    protocol_cols=protocol_cols
)

In [ ]:
# Filtered and unfiltered solutions 
for ct in filtered_df:
    print(f"Filtered {ct} number of solutions:", filtered_df[ct].shape[0])
    print(f"Unfiltered {ct} number of solutions:", pure_cell_type_solution_dict_concat[ct].shape[0], "\n")

In [ ]:
filtered_df["late_MgkPro"]["late_MgkPro_prop"].max()

In [ ]:
import pickle as pkl

for cell_type in filtered_df:
    # Clean filtered data frames 
    filtered_df[cell_type]= filtered_df[cell_type].rename({"target_ct_loss_std": "uncertainty_loss", 
                                                           "ct_prop_total_variance": "uncertainty_total_variance_ct"}, 
                                                          axis=1)
    
    filtered_df[cell_type] = filtered_df[cell_type].drop(["cfg:constraints:c_scheduler_kwargs:t_warmup",
                                                          "cfg:constraints:c_scheduler_kwargs:vmin",
                                                          "cfg:constraints:c_scheduler_kwargs:vmax"], axis=1)

    # Clean unfiltered dataframes 
    annotated_df[cell_type]= annotated_df[cell_type].rename({"target_ct_loss_std": "uncertainty_loss", 
                                                           "ct_prop_total_variance": "uncertainty_total_variance_ct"}, 
                                                          axis=1)
    
    annotated_df[cell_type] = annotated_df[cell_type].drop(["cfg:constraints:c_scheduler_kwargs:t_warmup",
                                                          "cfg:constraints:c_scheduler_kwargs:vmin",
                                                          "cfg:constraints:c_scheduler_kwargs:vmax"], axis=1)

    # Clean unfiltered dataframes 
    pure_cell_type_solution_dict_concat[cell_type]= pure_cell_type_solution_dict_concat[cell_type].rename({"target_ct_loss_std": "uncertainty_loss", 
                                                           "ct_prop_total_variance": "uncertainty_total_variance_ct"}, 
                                                          axis=1)
    
    pure_cell_type_solution_dict_concat[cell_type] = pure_cell_type_solution_dict_concat[cell_type].drop(["cfg:constraints:c_scheduler_kwargs:t_warmup",
                                                          "cfg:constraints:c_scheduler_kwargs:vmin",
                                                          "cfg:constraints:c_scheduler_kwargs:vmax"], axis=1)

## Plot cell type proportions vs value of the parameter 

In [ ]:
# sns.set(style="whitegrid", context="paper")
plt.figure(figsize=(3,3))
df_plot = filtered_df["late_MgkPro"].reset_index(drop=True)

ax = sns.scatterplot(
    data=df_plot,
    x=f"late_MgkPro_prop",
    y="uncertainty_loss",
    color="dimgray",
    legend=False,
    s=20,
    alpha=0.6,
    edgecolor=None
)

ax.set_title(cell_type)
ax.set_xlabel("Cell type proportion")
ax.set_ylabel("Loss uncertainty")

sns.despine(ax=ax)

In [ ]:
sorted_mgkpro = filtered_df["late_MgkPro"].sort_values(["late_MgkPro_prop"], ascending=False).reset_index(drop=True)

In [ ]:
for col in sorted_mgkpro.columns: 
    print(col)

In [ ]:
sorted_mgkpro.loc[:, ["gm-csf_[ng_ml]",
                      "tpo_[ng_ml]", 
                      "sr1_[nm]",
                      "um171_[nm]",
                      "um729_[µm]",
                      "scf_[ng_ml]",
                      "butyzamide_[nm]",
                      "retinoic_acid_[µm]",
                      "ldl_[ng_ml]",
                      "il3_[ng_ml]",
                      "o2_[%]",
                      "days_of_culture", 
                     "late_MgkPro_prop"]]

In [ ]:
# sns.clustermap(
#     np.log1p(sorted_mgkpro.loc[:, protocol_cols].astype(float)),
#     row_cluster=True,
#     col_cluster=False,
#     figsize=(6, 6)
# )

# # Sort rows by late_MgkPro_prop before plotting
# sorted_df = sorted_mgkpro.sort_values(
#     by="late_MgkPro_prop",
#     ascending=True
# )

sorted_df = sorted_mgkpro.sort_values(
    by="uncertainty_loss",
    ascending=False
)

# Create clustermap
g = sns.clustermap(
    np.log1p(sorted_df[:20].loc[:, protocol_cols].astype(float)),
    row_cluster=True,
    col_cluster=False,
    figsize=(6, 4), 
    cmap="Grays"
)

# Increase tick label sizes
g.ax_heatmap.tick_params(axis='x', labelsize=12)
g.ax_heatmap.tick_params(axis='y', labelsize=12)

# Optional: rotate x labels for readability
plt.setp(
    g.ax_heatmap.get_xticklabels(),
    rotation=90,
    ha="right"
)

# Save as SVG
g.savefig(
    "plots/mgkpro_clustermap.png",
    format="png",
    bbox_inches="tight", dpi=500, 
)
# sns.clustermap(
#     np.log1p(sorted_df[:20].sort_values("late_MgkPro_prop", ascending=True).loc[:, protocol_cols].astype(float)),
#     row_cluster=False,   # preserve sorting
#     col_cluster=False,
#     figsize=(6, 6)
# )

# Plot Sakurai pattern

In [ ]:
sakurai_pattern = np.log1p(np.array([0, 0, 0, 70, 0, 0, 100, 0, 0, 0, 20, 14]))

In [ ]:
adata_original = sc.read_h5ad("../../../../../../project_folder/output/loops/data/loop0/adata_500k.h5ad")
protocol_obs = adata_original.obs.loc[:, protocol_cols].astype(float)

In [ ]:
protocol_unique = {protocol: np.unique(protocol_obs[protocol]) for protocol in protocol_obs.columns}

In [ ]:
mgkpro_filtered = filtered_df["late_MgkPro"].loc[:, protocol_cols]

In [ ]:
for col in mgkpro_filtered.columns:
    nn_col = ((mgkpro_filtered[col].values[:, None] - protocol_unique[col][None, :])**2)
    nn_col = nn_col.argmin(-1)
    assigned = protocol_unique[col][nn_col]
    mgkpro_filtered[col] = assigned

In [ ]:
mgkpro_filtered = mgkpro_filtered.drop(columns=["days_of_culture"], axis=1)

In [ ]:
# filtered_df["late_MgkPro"]["is_sakurai"]  = (mgkpro_filtered==sakurai_pattern[None, :]).all(axis=1)

In [ ]:
plt.figure(figsize=(4,4))

# Plot gray (background) points first
sns.scatterplot(
    data=filtered_df["late_MgkPro"][filtered_df["late_MgkPro"]["is_sakurai"] == False],
    x="late_MgkPro_prop",
    y="uncertainty_loss",
    color="lightgray",
    s=140,
    alpha=0.6,
    edgecolor="none",
    legend=False
)

# Plot red (highlighted) points on top
sns.scatterplot(
    data=filtered_df["late_MgkPro"][filtered_df["late_MgkPro"]["is_sakurai"] == True],
    x="late_MgkPro_prop",
    y="uncertainty_loss",
    color="firebrick",
    s=140,
    alpha=0.9,
    edgecolor="none",
    legend=False
)

plt.grid(False)
plt.xlabel("", fontsize=16)
plt.ylabel("", fontsize=16)
plt.tick_params(axis='both', which='major', labelsize=14, width=1.5)

# Clean up spines to match figure style
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_color('black')
plt.tick_params(axis='both', which='major', labelsize=14, width=1.5, colors='black')

# Custom legend for just the highlighted group
red_dot = plt.Line2D([], [], marker='o', color='firebrick', linestyle='None',
                      markersize=12, alpha=0.9)
# plt.legend([red_dot], ["Similar to Exp. 210"], loc="upper right", frameon=True, fontsize=13)

plt.tight_layout()
plt.savefig("plots/sakurai_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

# Sakurai pattern heatmap 

In [ ]:
sakurai_pattern_df = pd.DataFrame(sakurai_pattern)

In [ ]:
plt.figure(figsize=(7,0.5))
sns.heatmap(sakurai_pattern_df.T, cmap="Greys")
plt.savefig("plots/sakurai_pattern.png")